# 1. Download required packages, make sure Ultralytics is 8.4+ (latest)

In [1]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00a 0:00:01


In [ ]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

# WIDER_FACE/
# ├── WIDER_train/images/...
# ├── WIDER_val/images/...
# └── wider_face_split/wider_face_train_bbx_gt.txt, ...

DATASET_ROOT = '/kaggle/input/datasets/revdra/wider-face' 
OUTPUT_DIR = 'datasets/widerface_yolo'

def convert_box(size, box):
    # box: x, y, w, h
    dw = 1. / size[1]
    dh = 1. / size[0]
    x = box[0] + box[2] / 2.0
    y = box[1] + box[3] / 2.0
    w = box[2]
    h = box[3]
    
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)

def process_set(set_name, split_file, img_root):
    save_img_dir = Path(OUTPUT_DIR) / 'images' / set_name
    save_label_dir = Path(OUTPUT_DIR) / 'labels' / set_name
    save_img_dir.mkdir(parents=True, exist_ok=True)
    save_label_dir.mkdir(parents=True, exist_ok=True)

    with open(split_file, 'r') as f:
        lines = f.readlines()

    idx = 0
    # Use tqdm to show progress
    pbar = tqdm(total=len(lines), desc=f"Processing {set_name}")
    
    while idx < len(lines):
        line = lines[idx].strip()
        
        # --- FIX ROBUSTNESS: Skip blank lines or not image ---
        if not line.endswith('.jpg'):
            idx += 1
            pbar.update(1)
            continue
            
        filename = line
        idx += 1
        
        # Đọc số lượng box
        try:
            num_boxes = int(lines[idx].strip())
        except ValueError:
            # resync
            idx += 1 
            pbar.update(1)
            continue
            
        idx += 1
        
        boxes = []
        if num_boxes == 0:

            if idx < len(lines):
                next_line = lines[idx].strip()
                if not next_line.endswith('.jpg'):
                    idx += 1 # Skip dummy
        else:
            for _ in range(num_boxes):
                if idx >= len(lines): break
                # x1, y1, w, h, ...
                val_strs = lines[idx].strip().split()
                # Take only the first 4 values for box (ignore blur, expression, etc.)
                if len(val_strs) >= 4:
                    boxes.append(list(map(float, val_strs[:4])))
                idx += 1

        # Update progress bar (1 for filename line, 1 for num_boxes line, and num_boxes for box lines)
        # (filename line + num_boxes line + box lines)
        pbar.update(2 + (1 if num_boxes == 0 else num_boxes))

        # --- COPY IMG & ADD LABEL ---
        src_img_path = os.path.join(img_root, filename)
        
        # Check if image exists before processing
        if not os.path.exists(src_img_path):
            continue
            
        img = cv2.imread(src_img_path)
        if img is None: continue
        h, w = img.shape[:2]

        dst_img_path = save_img_dir / Path(filename).name
        shutil.copy(src_img_path, dst_img_path)

        # Write file label (if box)
        if len(boxes) > 0:
            label_file = save_label_dir / (Path(filename).stem + '.txt')
            with open(label_file, 'w') as lf:
                for box in boxes:
                    # box: x1, y1, w, h
                    # Convert to format YOLO: x_center, y_center, w, h (normalized)
                    xywh = convert_box((h, w), box)
                    
                    # Limit values to [0, 1]
                    xc, yc, bw, bh = xywh
                    xc = max(0, min(1, xc))
                    yc = max(0, min(1, yc))
                    bw = max(0, min(1, bw))
                    bh = max(0, min(1, bh))
                    
                    lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

if __name__ == "__main__":
    # Train set
    process_set('train', 
                os.path.join(DATASET_ROOT, 'wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt'),
                os.path.join(DATASET_ROOT, 'WIDER_train/WIDER_train/images'))
    # Val set
    process_set('val', 
                os.path.join(DATASET_ROOT, 'wider_face_split/wider_face_split/wider_face_val_bbx_gt.txt'),
                os.path.join(DATASET_ROOT, 'WIDER_val/WIDER_val/images'))
    
    # Create data.yaml for YOLO
    yaml_content = f"""
path: {os.path.abspath(OUTPUT_DIR)}
train: images/train
val: images/val
test: images/val  # WIDER FACE test set không có public labels, ta dùng val để test
names:
  0: face
"""
    with open(f"{OUTPUT_DIR}/data.yaml", "w") as f:
        f.write(yaml_content)
    
    print("Dataset conversion complete.")

Processing val: 100%|██████████| 46160/46160 [00:35<00:00, 1308.22it/s]

Dataset conversion complete.


In [ ]:
# 1. Train Nano
!yolo train model=/kaggle/input/models/revdra/yolov12/pytorch/default/1/yolov12n.pt data=/kaggle/working/datasets/widerface_yolo/data.yaml epochs=50 imgsz=640 batch=32 device=0,1 project=runs/train name=yolov12n exist_ok=True

In [ ]:
# 2. Train Small
!yolo train model=/kaggle/input/models/revdra/yolov12/pytorch/default/1/yolov12s.pt data=/kaggle/working/datasets/widerface_yolo/data.yaml epochs=50 imgsz=640 batch=32 device=0,1 project=runs/train name=yolov12s exist_ok=True

In [ ]:
# 3. Train Medium
!yolo train model=/kaggle/input/models/revdra/yolov12/pytorch/default/1/yolov12m.pt data=/kaggle/working/datasets/widerface_yolo/data.yaml epochs=50 imgsz=640 batch=16 device=0,1 project=runs/train name=yolov12m exist_ok=True

In [5]:
# 4. Train Large
!yolo detect train model=yolo12l.pt data=/kaggle/working/datasets/widerface_yolo/data.yaml epochs=50 imgsz=640 batch=8 project=runs/train name=yolov12l exist_ok=True device =0,1

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/datasets/widerface_yolo/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov12l, nbs=64